# Long-Short Term Memory and Fully Convolutional Network (LSTMFCN) for Classifying Human Activities


In [ ]:
from zipfile import ZipFile
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
with ZipFile('/content/human+activity+recognition+using+smartphones.zip') as zObj:
    with zObj.open(zObj.filelist[1]) as inner_zip:
        with ZipFile(inner_zip) as inner:
            inner.extractall('/content/')

In [ ]:
# each window lasts for 2.56 seconds.
# 2.56 seconds * 50 Hz = 128 reading per window
2.56*50

128.0

In [ ]:
root_path = '/content/UCI HAR Dataset/train/Inertial Signals/'
X_train = {}
for f in os.listdir(root_path):
    X_train[f] = np.loadtxt(root_path + f)

X_train = dict(sorted(X_train.items()))
X_train = np.stack(list(X_train.values()), axis=1)
X_train.shape

(7352, 9, 128)

In [ ]:
root_path = '/content/UCI HAR Dataset/test/Inertial Signals/'
X_test = {}
for f in os.listdir(root_path):
    X_test[f] = np.loadtxt(root_path + f)

X_test = dict(sorted(X_test.items()))
X_test = np.stack(list(X_test.values()), axis=1)
X_test.shape

(2947, 9, 128)

In [ ]:
y_train = np.loadtxt('/content/UCI HAR Dataset/train/y_train.txt')
y_test = np.loadtxt('/content/UCI HAR Dataset/test/y_test.txt')
assert(len(y_train) == len(X_train))
assert(len(y_test) == len(X_test))

In [ ]:
# get the activities labels
activities = np.loadtxt('/content/UCI HAR Dataset/activity_labels.txt', usecols=1, dtype='str')

# Data visualization

In [ ]:
_, axes = plt.subplots(3 ,2, tight_layout=True, figsize=(8,8))
for i, ax in zip([9, 900, 1300, 3300, 4800, 6600], axes.flatten()):
    ax.plot(X_train[i].T)
    ax.set_title(f'Sliding window {i}. Label: {activities[int(y_train[i]-1)]}');

In [ ]:
_, ax = plt.subplots(figsize=(4,3))
ax.barh(np.arange(1,7), np.unique(y_train, return_counts=True)[1], tick_label=activities, color='#3a548c');

# Model

In [ ]:
# from github https://github.com/titu1994/MLSTM-FCN/blob/master/net_flow_model.py
# I modified the code a bit, especially the total filters/kernels
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Attention, multiply, concatenate, Activation, Masking, Reshape
from tensorflow.keras.layers import Conv1D, BatchNormalization, GlobalAveragePooling1D, Permute, Dropout

def generate_model(input_shape:tuple, n_class:int,
                   kernels:list[int]=[8, 5, 3], filters:list[int]=[128, 256, 128],
                   lstm_cells:int=8, attention:bool=False):
    '''
    Implementation of MLSTM-FCN.
    Arguments:
        input_shape: Tuple, the shape of each multivariate time series
                     The shape expected is (n_variables, length_ts)
        n_class: int, the total classes
        kernels: List of 3 kernel sizes, one for each block. Defaults to [8, 5, 3]
        filters: List of 3 kernel numbers, one for each block. Defaults to [128, 256, 128]
        lstm_cells: int, the length of an LSTM layer. Defaults to 8
        attention: bool, whether the Attention layer is used or not.
                   Setting the parameter to True means that we apply MALSTM-FCN
    '''
    ip = Input(shape=input_shape)

    x = Masking()(ip)

    if attention:
        x = Attention()([ip, ip], use_causal_mask=True)
        x = LSTM(lstm_cells)(x)
    else:
        x = LSTM(lstm_cells)(x)
    x = Dropout(0.8)(x)

    y = Permute((2, 1))(ip)
    y = Conv1D(filters[0], kernels[0], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[1], kernels[1], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[2], kernels[2], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)

    y = GlobalAveragePooling1D()(y)

    x = concatenate([x, y])

    out = Dense(n_class, activation='softmax')(x)

    model = Model(ip, out)
    model.summary()

    # add load model code here to fine-tune

    return model

def generate_model_no_permute(input_shape:tuple, n_class:int,
                   kernels:list[int]=[8, 5, 3], filters:list[int]=[128, 256, 128],
                   lstm_cells:int=8, attention:bool=False):
    '''
    Implementation of MLSTM-FCN without shuffling dimension.
    Arguments:
        input_shape: Tuple, the shape of each multivariate time series
                     The shape expected is (length_ts, n_variables)
        n_class: int, the total classes
        kernels: List of 3 kernel sizes, one for each block. Defaults to [8, 5, 3]
        filters: List of 3 kernel numbers, one for each block. Defaults to [128, 256, 128]
        lstm_cells: int, the length of an LSTM layer. Defaults to 8
        attention: bool, whether the Attention layer is used or not.
                   Setting the parameter to True means that we apply MALSTM-FCN
    '''
    ip = Input(shape=input_shape)

    x = Masking()(ip)

    if attention:
        x = Attention()([ip, ip], use_causal_mask=True)
        x = LSTM(lstm_cells)(x)
    else:
        x = LSTM(lstm_cells)(x)
    x = Dropout(0.8)(x)

    # y = Permute((2, 1))(ip)
    y = Conv1D(filters[0], kernels[0], padding='same', kernel_initializer='he_uniform')(ip)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[1], kernels[1], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[2], kernels[2], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)

    y = GlobalAveragePooling1D()(y)

    x = concatenate([x, y])

    out = Dense(n_class, activation='softmax')(x)

    model = Model(ip, out)
    model.summary()

    # add load model code here to fine-tune

    return model

def squeeze_excite_block(input):
    ''' Create a squeeze-excite block
    Args:
        input: input tensor
        filters: number of output filters
        k: width factor

    Returns: a keras tensor
    '''
    filters = input.shape[-1] # channel_axis = -1 for TF

    se = GlobalAveragePooling1D()(input)
    se = Reshape((1, filters))(se)
    se = Dense(filters // 16,  activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
    se = Dense(filters, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)
    se = multiply([input, se])
    return se

# Applying MLSTM-FCN classifier

In [ ]:
config = {'n_class':len(activities),
          'kernels':[64, 20, 4], 'filters':[64, 128, 64],
          'lstm_cells':40, 'attention':False}

# set how many times we want to fit the model
FIT_TIMES = 10

In [ ]:
preds = {}
for i in range(FIT_TIMES):
    print(f'{i}th iteration starts -------------------------')
    tf.keras.backend.clear_session()
    model = generate_model(input_shape=(9, 128), **config)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy',
                metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy', 'f1_score'])
    model.fit(X_train, pd.get_dummies(y_train), epochs=8, verbose=2, batch_size=128)
    preds[f'pred_{i}'] = model.predict(X_test)
    model.save(f'model_{i}.keras')

# Results

In [ ]:
def metric_report(metric_values: list, range:bool=False) -> float:
    '''print out the metric values to the thousandths precision.
       Example:
           0.995331 ----> 0.995
           0.97482  ----> 0.975
       Sometimes I train a model more than once to see the robustness of the model,
       and when I do that, I use a list to store the metrics from all iterations.
       This function is to make printing out the summary easier.

       Parameters:
           metric_values: a list of floats
           range: bool. If True, then the range of metric_values is returned.
    '''
    # if range is True, that means I want to get the range of the array.
    if range:
        max = np.max(metric_values)
        min = np.min(metric_values)
        output = max-min
    # if range is False, I want to get the average of the array.
    else:
        output = np.mean(metric_values)
    return np.round(output, 3)

In [ ]:
aucs = []
y_test_dummies = pd.get_dummies(y_test)

for i in range(FIT_TIMES):
    aucs.append(roc_auc_score(y_test_dummies,
                              preds[f'pred_{i}'],
                              multi_class='ovo'))

plt.plot(np.arange(FIT_TIMES), aucs, '-o', color='#481f70')
plt.axhline(y=np.mean(aucs), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('AUC score')
plt.title(f'average AUC: {metric_report(aucs)}\n'
          f'range AUC: {metric_report(aucs, range=True)}');

In [ ]:
accuracies = []
for i in range(FIT_TIMES):
    accuracies.append(accuracy_score(y_test, 1 + np.argmax(preds[f'pred_{i}'], axis=1)))

plt.plot(np.arange(FIT_TIMES), accuracies, '-o', color='#481f70')
plt.axhline(y=np.mean(accuracies), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('accuracy (%)')
plt.title(f'average accuracy: {metric_report(accuracies)}\n'
          f'accuracy range: {metric_report(accuracies, range=True)}');

In [ ]:
# taking hard predictions from the best model
best_model = np.argmax(accuracies)
y_pred = 1+ np.argmax(preds[f'pred_{best_model}'], axis=1)

# visualizing the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels= activities)
plt.tick_params(axis='x', rotation=90)
plt.show();

In [ ]:
best_model

## loading the saved model

In [ ]:
loaded_model = keras.models.load_model(f'model_{best_model}.keras')

In [ ]:
loaded_model.compute_metrics(X_test, y_test_dummies, preds[f'pred_{best_model}'])

{'accuracy': 0.9216151833534241,
 'auc': 0.9944837689399719,
 'f1_score': array([0.96265554, 0.97033894, 0.95553017, 0.8051391 , 0.83923703,
        1.        ], dtype=float32),
 'loss': 0.0}

In [ ]:
predict_again = loaded_model.predict(X_test)
print(predict_again.shape)
np.array_equal(preds[f'pred_{best_model}'], predict_again)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
(2947, 6)


True

In [ ]:
loaded_model.weights[26]

<Variable path=dense_4/bias, shape=(6,), dtype=float32, value=[-0.00709399 -0.00810799  0.00303072  0.01060226  0.01118769 -0.01362943]>

# EXPERIMENT: LSTM-FCN without dimension shuffle

In [ ]:
root_path = '/content/UCI HAR Dataset/train/Inertial Signals/'
X_train = {}
for f in os.listdir(root_path):
    X_train[f] = np.loadtxt(root_path + f)

X_train = dict(sorted(X_train.items()))
X_train = np.stack(list(X_train.values()), axis=-1)
X_train.shape

(7352, 128, 9)

In [ ]:
root_path = '/content/UCI HAR Dataset/test/Inertial Signals/'
X_test = {}
for f in os.listdir(root_path):
    X_test[f] = np.loadtxt(root_path + f)

X_test = dict(sorted(X_test.items()))
X_test = np.stack(list(X_test.values()), axis=-1)
X_test.shape

(2947, 128, 9)

In [ ]:
preds_no_permute = {}
for i in range(FIT_TIMES):
    print(f'{i}th iteration starts -------------------------')
    tf.keras.backend.clear_session()
    model = generate_model_no_permute(input_shape=(128, 9), **config)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy',
                metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy', 'f1_score'])
    model.fit(X_train, pd.get_dummies(y_train), epochs=8, verbose=2, batch_size=128)
    preds_no_permute[f'pred_{i}'] = model.predict(X_test)
    model.save(f'model_no_permute_{i}.keras')

In [ ]:
aucs_no_permute = []
# y_test_dummies = pd.get_dummies(y_test)

for i in range(FIT_TIMES):
    aucs_no_permute.append(roc_auc_score(y_test_dummies,
                              preds_no_permute[f'pred_{i}'],
                              multi_class='ovo'))

plt.plot(np.arange(FIT_TIMES), aucs_no_permute, '-o', color='#481f70')
plt.axhline(y=np.mean(aucs_no_permute), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('AUC score of model w/o permute')
plt.title(f'average AUC: {metric_report(aucs_no_permute)}\n'
          f'range AUC: {metric_report(aucs_no_permute, range=True)}');

In [ ]:
accuracies_no_permute = []
for i in range(FIT_TIMES):
    accuracies_no_permute.append(accuracy_score(y_test, 1 + np.argmax(preds_no_permute[f'pred_{i}'], axis=1)))

plt.plot(np.arange(FIT_TIMES), accuracies_no_permute, '-o', color='#481f70')
plt.axhline(y=np.mean(accuracies_no_permute), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('accuracy (%)')
plt.title(f'average accuracy: {metric_report(accuracies_no_permute)}\n'
          f'accuracy range: {metric_report(accuracies_no_permute, range=True)}');

In [ ]:
# taking hard predictions from the best model
best_model = np.argmax(accuracies_no_permute)
y_pred = 1+ np.argmax(preds[f'pred_{best_model}'], axis=1)

# visualizing the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels= activities)
plt.tick_params(axis='x', rotation=90)
plt.show();

# EXPERIMENT: reshape the data (dimension shuffling on the convolutional part)

calling the `generate_model` function, but with the transposed input data (which is the data in the previous section)

In [ ]:
preds_reshape = {}
models_reshape = {}
for i in range(FIT_TIMES):
    print(f'{i}th iteration starts -------------------------')
    tf.keras.backend.clear_session()
    model = generate_model(input_shape=(128, 9), **config)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy',
                metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy', 'f1_score'])
    model.fit(X_train, pd.get_dummies(y_train), epochs=8, verbose=2, batch_size=128)
    preds_reshape[f'pred_{i}'] = model.predict(X_test)

In [ ]:
aucs_reshape = []
# y_test_dummies = pd.get_dummies(y_test)

for i in range(FIT_TIMES):
    aucs_reshape.append(roc_auc_score(y_test_dummies,
                              preds_reshape[f'pred_{i}'],
                              multi_class='ovo'))

plt.plot(np.arange(FIT_TIMES), aucs_reshape, '-o', color='#481f70')
plt.axhline(y=np.mean(aucs_reshape), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('AUC score')
plt.title(f'average AUC: {metric_report(aucs_reshape)}\n'
          f'range AUC: {metric_report(aucs_reshape, range=True)}');

In [ ]:
accuracies_reshape = []
for i in range(FIT_TIMES):
    accuracies_reshape.append(accuracy_score(y_test, 1 + np.argmax(preds_reshape[f'pred_{i}'], axis=1)))

plt.plot(np.arange(FIT_TIMES), accuracies_reshape, '-o', color='#481f70')
plt.axhline(y=np.mean(accuracies_reshape), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('accuracy (%)')
plt.title(f'average accuracy: {metric_report(accuracies_reshape)}\n'
          f'accuracy range: {metric_report(accuracies_reshape, range=True)}');

# EXPERIMENT: fully connected layer

Will a Dense layer be able to replace the LSTM layer?

I doubt it, but let's see.

In [ ]:
def generate_model_con(input_shape:tuple, n_class:int,
                   kernels:list[int]=[8, 5, 3], filters:list[int]=[128, 256, 128],
                   lstm_cells:int=8, attention:bool=False):
    '''
    Implementation of MLSTM-FCN.
    Arguments:
        input_shape: Tuple, the shape of each multivariate time series
                     The shape expected is (n_variables, length_ts)
        n_class: int, the total classes
        kernels: List of 3 kernel sizes, one for each block. Defaults to [8, 5, 3]
        filters: List of 3 kernel numbers, one for each block. Defaults to [128, 256, 128]
        lstm_cells: int, the length of an LSTM layer. Defaults to 8
        attention: bool, whether the Attention layer is used or not.
                   Setting the parameter to True means that we apply MALSTM-FCN
    '''
    ip = Input(shape=input_shape)

    x = Masking()(ip)

    x = Dense(lstm_cells)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.8)(x)

    y = Permute((2, 1))(ip)
    y = Conv1D(filters[0], kernels[0], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[1], kernels[1], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)
    y = squeeze_excite_block(y)

    y = Conv1D(filters[2], kernels[2], padding='same', kernel_initializer='he_uniform')(y)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)

    y = GlobalAveragePooling1D()(y)

    x = concatenate([x, y])

    out = Dense(n_class, activation='softmax')(x)

    model = Model(ip, out)
    model.summary()

    # add load model code here to fine-tune

    return model

In [ ]:
preds_con = {}
for i in range(FIT_TIMES):
    print(f'{i}th iteration starts -------------------------')
    tf.keras.backend.clear_session()
    model = generate_model_con(input_shape=(9, 128), **config)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy',
                metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy', 'f1_score'])
    model.fit(X_train, pd.get_dummies(y_train), epochs=8, verbose=2, batch_size=128)
    preds_con[f'pred_{i}'] = model.predict(X_test)

In [ ]:
aucs = []
y_test_dummies = pd.get_dummies(y_test)

for i in range(FIT_TIMES):
    aucs.append(roc_auc_score(y_test_dummies,
                              preds_con[f'pred_{i}'],
                              multi_class='ovo'))

plt.plot(np.arange(FIT_TIMES), aucs, '-o', color='#481f70')
plt.axhline(y=np.mean(aucs), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('AUC score')
plt.title(f'average AUC: {metric_report(aucs)}\n'
          f'range AUC: {metric_report(aucs, range=True)}');

In [ ]:
accuracies = []
for i in range(FIT_TIMES):
    accuracies.append(accuracy_score(y_test, 1 + np.argmax(preds_con[f'pred_{i}'], axis=1)))

plt.plot(np.arange(FIT_TIMES), accuracies, '-o', color='#481f70')
plt.axhline(y=np.mean(accuracies), color='grey')
plt.xlabel('n_iteration')
plt.ylabel('accuracy (%)')
plt.title(f'average accuracy: {metric_report(accuracies)}\n'
          f'accuracy range: {metric_report(accuracies, range=True)}');

# References

Reference to the data:

---

Davide Anguita, Alessandro Ghio, Luca Oneto, Xavier Parra and Jorge L. Reyes-Ortiz. A Public Domain Dataset for Human Activity Recognition Using Smartphones. 21th European Symposium on Artificial Neural Networks, Computational Intelligence and Machine Learning, ESANN 2013. Bruges, Belgium 24-26 April 2013.
<br> <br> <br>
Reference to the paper:

---

Karim, F., Majumdar, S., Darabi, H., & Harford, S. (2019). Multivariate LSTM-FCNs for time series classification. *Neural Networks, 116, 237–245.*
